# Exploratory Data Analysis — Industrial Safety & Asset Monitoring Dataset

**Dissertation:** Edge AI Vision Pipeline on AWS SageMaker for Real-Time Object Detection
**Institution:** Liverpool John Moores University — MSc Embedded Systems and IC Design
**Author:** Tanu Sharma

This notebook documents the exploratory analysis performed on the assembled 9,600-image
industrial dataset before training began — class distribution, bounding-box size
distribution (which directly motivated the anchor re-clustering approach in
`training/anchor_clustering.py`), and image-quality checks that informed the
augmentation pipeline design in `data_pipeline/augmentation.py`.

See `docs/methodology.md` for the full dataset composition and split-strategy rationale.


In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

DATASET_ROOT = Path("../data")
CLASS_SCHEMA = [
    "person_ppe_compliant",
    "person_ppe_violation",
    "forklift",
    "fixed_machinery",
    "restricted_zone_marker",
]


## 1. Class Distribution

Loads the per-class instance counts computed during
`data_pipeline/annotation_conversion.py`'s RecordIO build pass.

In [ ]:
with open(DATASET_ROOT / "class_distribution.json") as f:
    class_counts = json.load(f)

df_classes = pd.DataFrame(
    {"class": list(class_counts.keys()), "instances": list(class_counts.values())}
)
df_classes["share_pct"] = 100 * df_classes["instances"] / df_classes["instances"].sum()
df_classes.sort_values("instances", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=df_classes, x="instances", y="class", ax=ax)
ax.set_title("Instance count per class (9,600-image dataset)")
ax.set_xlabel("Instance count")
ax.set_ylabel("")
for i, row in df_classes.iterrows():
    ax.text(row["instances"] + 20, i, f"{row['share_pct']:.1f}%", va="center")
plt.tight_layout()
plt.savefig("../docs/figures/class_distribution.png", dpi=150)
plt.show()


**Observation:** `restricted_zone_marker` accounts for only 4.2% of labelled
instances — a substantial class imbalance that directly motivated (a) the stratified
split strategy in `data_pipeline/dataset_split.py`, which enforces a minimum
per-class test-set instance count, and (b) the hard-negative mining pass added to
training for this specific class (`training/train_yolo3_sagemaker.py`).

## 2. Bounding Box Size Distribution

This is the analysis that directly motivated re-clustering YOLOv3's anchor boxes
rather than reusing COCO's defaults — see `docs/model_architecture.md#anchor-re-clustering`.

In [ ]:
with open(DATASET_ROOT / "train_box_dimensions.json") as f:
    box_data = json.load(f)

boxes = np.array(box_data["boxes"])  # normalised [w, h] pairs
box_classes = np.array(box_data["classes"])

df_boxes = pd.DataFrame({
    "width_norm": boxes[:, 0],
    "height_norm": boxes[:, 1],
    "area_norm": boxes[:, 0] * boxes[:, 1],
    "aspect_ratio": boxes[:, 0] / boxes[:, 1],
    "class": [CLASS_SCHEMA[c] for c in box_classes],
})
df_boxes.groupby("class")[["width_norm", "height_norm", "aspect_ratio"]].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(data=df_boxes.sample(2000, random_state=13),
                 x="width_norm", y="height_norm", hue="class", alpha=0.5, ax=axes[0])
axes[0].set_title("Normalised box width vs. height by class")

sns.histplot(data=df_boxes, x="area_norm", hue="class", element="step",
             log_scale=(True, False), ax=axes[1])
axes[1].set_title("Normalised box area distribution by class")

plt.tight_layout()
plt.savefig("../docs/figures/box_size_distribution.png", dpi=150)
plt.show()


**Observation:** the width/height scatter shows a visibly different clustering
structure than COCO's natural-image object distribution — industrial CCTV's fixed,
elevated camera geometry produces a narrower aspect-ratio range per class. This is the
empirical basis for running k-means anchor re-clustering directly on this
distribution (`training/anchor_clustering.py`) rather than reusing COCO's anchor
priors, which was the single highest-impact change identified in the later ablation
study (`docs/evaluation_and_results.md#ablation-study`).

## 3. Image Quality / Lighting Condition Audit

A manual audit of a 400-image stratified sample, tagging each frame's dominant
lighting condition, informed the synthetic glare/low-light augmentation
(`data_pipeline/augmentation.py`).

In [ ]:
with open(DATASET_ROOT / "lighting_audit_sample.json") as f:
    lighting_audit = json.load(f)

df_lighting = pd.DataFrame(lighting_audit)
lighting_summary = df_lighting["condition"].value_counts(normalize=True) * 100
lighting_summary


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
lighting_summary.plot(kind="bar", ax=ax, color=sns.color_palette("deep"))
ax.set_ylabel("Share of audited sample (%)")
ax.set_title("Lighting condition distribution (400-image audit sample)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../docs/figures/lighting_audit.png", dpi=150)
plt.show()


**Observation:** roughly a third of the audited sample showed either glare
(skylight-lit loading bays) or shadowed regions — the empirical basis for including
`synthetic_glare` in the augmentation pipeline rather than assuming standard HSV
jitter alone would cover this failure mode. See
`docs/model_architecture.md#domain-specific-augmentation` for how each augmentation
choice traces back to a specific observed condition, including this one.

## 4. Summary

This EDA pass directly informed three downstream design decisions:

1. **Anchor re-clustering** (Section 2) — the dominant contributor in the later ablation study
2. **Hard-negative mining for `restricted_zone_marker`** (Section 1) — a direct response to the measured 4.2% instance share
3. **Synthetic glare/motion-blur augmentation** (Section 3) — targeted at specific, measured visual conditions rather than applied as generic defaults

Full methodology and how these decisions were subsequently validated: `docs/methodology.md` and `docs/evaluation_and_results.md`.
